# Data Quality & Reconciliation

Runs **after** the Gold layer is built and acts as a pipeline-stopping gate. Any CRITICAL check failure raises an error in the final cell, halting the pipeline.

**Checks performed:**
1. **DQ-01** — All 8 TPC-H source tables ingested into Bronze (present and non-empty)
2. **DQ-02** — Primary keys populated and unique in Silver and Gold dimensions
3. **DQ-03** — Gold `fact_sales` contains no null surrogate keys; grain is unique
4. **DQ-04** — Numeric measures satisfy documented business rules
5. **DQ-05** — Gold revenue and row counts reconcile with Silver (within tolerance)
6. **DQ-06** — Rejected-record counts reported per table (Bronze = Silver + Quarantine)

In [0]:
# ─── Configuration & check framework ────────────────────────────────────────────
# Data-quality gate for the TPC-H Sales & Supply Chain lakehouse.
# Runs AFTER the Gold layer is built. Any CRITICAL check failure raises a
# pipeline-stopping error in the final cell.

dbutils.widgets.text("CATALOG", "dev")
CATALOG = dbutils.widgets.get("CATALOG")

BRONZE = f"{CATALOG}.bronze"   # raw ingested source tables
SILVER = f"{CATALOG}.silver"   # cleaned tables + <table>_quarantine rejects
GOLD   = f"{CATALOG}.gold"     # fact / dim / kpi tables

RECON_TOLERANCE = 0.0001       # 0.01% relative tolerance for Gold vs Silver

# All eight TPC-H source tables that must exist in Bronze
SOURCE_TABLES = ["customer", "orders", "lineitem", "part",
                 "partsupp", "supplier", "nation", "region"]

# Silver table names (all match Bronze — no renaming needed in TPC-H)
SILVER_TABLE_NAMES = {t: t for t in SOURCE_TABLES}

# Primary keys per table (same column names in Bronze and Silver)
PRIMARY_KEYS = {
    "customer": ["c_custkey"],
    "orders":   ["o_orderkey"],
    "lineitem": ["l_orderkey", "l_linenumber"],   # composite grain key
    "part":     ["p_partkey"],
    "partsupp": ["ps_partkey", "ps_suppkey"],      # composite key
    "supplier": ["s_suppkey"],
    "nation":   ["n_nationkey"],
    "region":   ["r_regionkey"],
}

# Surrogate keys that must be populated on every Gold fact_sales row
FACT_SURROGATE_KEYS = [
    "order_key", "order_line_number",
    "customer_key", "product_key", "supplier_key",
    "customer_nation_key", "customer_region_key",
    "supplier_nation_key", "supplier_region_key",
]

# Gold dimension primary keys
GOLD_DIM_KEYS = {
    "dim_customer":   "customer_key",
    "dim_product":    "product_key",
    "dim_supplier":   "supplier_key",
    "dim_geography":  "geography_key",
}

import pyspark.sql.functions as F

check_results = []   # every check registers here; summarised in the final cell

def record_check(check_id, description, layer, critical, passed, details):
    """Register a check outcome. CRITICAL failures stop the pipeline at the end."""
    check_results.append({
        "check_id": check_id, "description": description, "layer": layer,
        "critical": critical, "passed": bool(passed), "details": details,
    })
    status = "PASS" if passed else "FAIL"
    tag = "CRITICAL" if critical else "INFO"
    print(f"[{status} | {tag}] {check_id} — {description}")
    print(f"    {details}\n")

def key_stats(schema, table, keys):
    """Return (row_count, null_key_rows, duplicate_key_values) for a table's key columns."""
    df = spark.table(f"{schema}.{table}")
    total = df.count()
    if total == 0:
        return 0, 0, 0
    nulls = df.filter(" OR ".join(f"{c} IS NULL" for c in keys)).count()
    dups = df.groupBy(*keys).count().filter(F.col("count") > 1).count()
    return total, nulls, dups

def rel_diff(a, b):
    """Relative difference between two numbers (0.0 when both are zero)."""
    a, b = float(a), float(b)
    if b == 0:
        return 0.0 if a == 0 else float("inf")
    return abs(a - b) / abs(b)

print(f"Data-quality gate initialised — catalog: {CATALOG}, reconciliation tolerance: {RECON_TOLERANCE:.4%}")
print(f"Bronze: {BRONZE} | Silver: {SILVER} | Gold: {GOLD}")

In [0]:
# ─── DQ-01 (critical): all eight source tables were ingested into Bronze ──────────
# Each table must exist in dev.bronze and have at least one row.

missing_tables, empty_tables, bronze_counts = [], [], {}

for t in SOURCE_TABLES:
    try:
        n = spark.table(f"{BRONZE}.{t}").count()
        bronze_counts[t] = n
        if n == 0:
            empty_tables.append(t)
    except Exception:
        missing_tables.append(t)

dq01_ok = not missing_tables and not empty_tables
record_check(
    "DQ-01", "All 8 source tables ingested into Bronze (present and non-empty)",
    "Bronze", critical=True, passed=dq01_ok,
    details=(f"ingested={len(bronze_counts)}/8"
             + (f", missing={missing_tables}" if missing_tables else "")
             + (f", empty={empty_tables}" if empty_tables else "")),
)

display(spark.createDataFrame(
    [(t, bronze_counts.get(t),
      "OK" if t in bronze_counts and bronze_counts[t] > 0 else "MISSING/EMPTY")
     for t in SOURCE_TABLES],
    "source_table string, bronze_rows long, status string",
))

In [0]:
# ─── DQ-02 (critical): primary keys populated and unique in Silver + Gold dims ────
# Bronze is the RAW landing zone — duplicate/null keys there are EXPECTED in dirty
# source data and are cleaned during Silver loading (rejects land in <table>_quarantine),
# so Bronze key health is reported as INFO only and never stops the pipeline.

# ── Silver PK checks (critical) ──
for t, keys in PRIMARY_KEYS.items():
    total, nulls, dups = key_stats(SILVER, SILVER_TABLE_NAMES[t], keys)
    record_check(
        "DQ-02a", f"Silver PK populated & unique — {t} ({', '.join(keys)})",
        "Silver", critical=True, passed=(nulls == 0 and dups == 0),
        details=f"rows={total:,}, null_key_rows={nulls}, duplicate_key_values={dups}",
    )

# ── Gold dimension key checks (critical) ──
for dim, key in GOLD_DIM_KEYS.items():
    total, nulls, dups = key_stats(GOLD, dim, [key])
    record_check(
        "DQ-02b", f"Gold dimension key populated & unique — {dim}.{key}",
        "Gold", critical=True, passed=(nulls == 0 and dups == 0),
        details=f"rows={total:,}, null_key_rows={nulls}, duplicate_key_values={dups}",
    )

# ── Bronze raw key health (informational only) ──
for t, keys in PRIMARY_KEYS.items():
    total, nulls, dups = key_stats(BRONZE, t, keys)
    record_check(
        "DQ-02c", f"Bronze raw key health (informational) — {t}",
        "Bronze", critical=False, passed=(nulls == 0 and dups == 0),
        details=(f"rows={total:,}, null_key_rows={nulls}, duplicate_key_values={dups} "
                 "(raw rejects are quarantined during Silver cleaning)"),
    )

In [0]:
# ─── DQ-03 (critical): Gold fact_sales contains no null surrogate keys ─────────────
# Every surrogate key column must be non-null on every fact row.
# The grain (order_key + order_line_number) must also be unique.

fact = spark.table(f"{GOLD}.fact_sales")
fact_rows = fact.count()

null_key_expr = " OR ".join(f"{c} IS NULL" for c in FACT_SURROGATE_KEYS)
null_surrogate_rows = fact.filter(null_key_expr).count() if fact_rows else 0

dup_grain = (fact.groupBy("order_key", "order_line_number").count()
             .filter(F.col("count") > 1).count()) if fact_rows else 0

record_check(
    "DQ-03a", "fact_sales has no null surrogate keys (incl. customer & supplier geography)",
    "Gold", critical=True, passed=(null_surrogate_rows == 0),
    details=f"fact_rows={fact_rows:,}, rows_with_null_surrogate_keys={null_surrogate_rows:,}",
)
record_check(
    "DQ-03b", "fact_sales grain unique (order_key + order_line_number)",
    "Gold", critical=True, passed=(dup_grain == 0),
    details=f"duplicate_grain_values={dup_grain}",
)

In [0]:
# ─── DQ-04 (critical): numeric measures satisfy the documented business rules ───
# TPC-H business rules:
#   quantity       > 0
#   extended_price  > 0
#   0 <= discount_rate <= 0.10   (TPC-H discount range: 0%–10%)
#   0 <= tax_rate     <= 0.08   (TPC-H tax range: 0%–8%)
#   net_revenue == extended_price * (1 - discount_rate)  (internal consistency)

fact = spark.table(f"{GOLD}.fact_sales")

bad_qty     = fact.filter("quantity IS NULL OR quantity <= 0").count()
bad_price   = fact.filter("extended_price IS NULL OR extended_price <= 0").count()
bad_disc    = fact.filter("discount_rate IS NULL OR discount_rate < 0 OR discount_rate > 0.10").count()
bad_tax     = fact.filter("tax_rate IS NULL OR tax_rate < 0 OR tax_rate > 0.08").count()
# net_revenue is stored as DECIMAL(38,4) but extended_price & discount_rate are
# DECIMAL(18,2); use a 1-cent absolute tolerance instead of exact equality to avoid
# false mismatches from decimal-precision promotion.
bad_netrev  = fact.filter(
    "ABS(net_revenue - (extended_price * (1 - discount_rate))) > 0.01"
).count()

dq04_ok = all(v == 0 for v in [bad_qty, bad_price, bad_disc, bad_tax, bad_netrev])
record_check(
    "DQ-04",
    "Business rules: quantity>0, extended_price>0, discount in [0,0.10], tax in [0,0.08], net_revenue consistent",
    "Gold", critical=True, passed=dq04_ok,
    details=(f"violations: quantity={bad_qty}, extended_price={bad_price}, "
             f"discount_rate={bad_disc}, tax_rate={bad_tax}, net_revenue_mismatch={bad_netrev}"),
)

In [0]:
# ─── DQ-05 (critical): Gold revenue and row counts reconcile with Silver ──────────
# The Silver side is computed INDEPENDENTLY from silver.lineitem (the source of
# truth for the fact grain) — never from a Gold table — so the reconciliation is real.
#
# Net revenue (before tax) = SUM(l_extendedprice × (1 − l_discount))

fact = spark.table(f"{GOLD}.fact_sales")
gold_rows = fact.count()
silver_rows = spark.table(f"{SILVER}.lineitem").count()

gold_rev = spark.sql(f"SELECT SUM(net_revenue) AS rev FROM {GOLD}.fact_sales").collect()[0][0] or 0
silver_rev = spark.sql(f"""
    SELECT SUM(CAST(l_extendedprice * (1 - l_discount) AS DECIMAL(18,2))) AS rev
    FROM {SILVER}.lineitem
""").collect()[0][0] or 0

row_diff = rel_diff(gold_rows, silver_rows)
rev_diff = rel_diff(gold_rev, silver_rev)

record_check(
    "DQ-05a", "Gold fact row count reconciles with Silver lineitem (<= 0.01%)",
    "Gold<->Silver", critical=True, passed=(row_diff <= RECON_TOLERANCE),
    details=f"gold_rows={gold_rows:,}, silver_rows={silver_rows:,}, rel_diff={row_diff:.6%}",
)
record_check(
    "DQ-05b", "Gold net revenue reconciles with independent Silver calculation (<= 0.01%)",
    "Gold<->Silver", critical=True, passed=(rev_diff <= RECON_TOLERANCE),
    details=(f"gold_net_revenue={float(gold_rev):,.2f}, "
             f"silver_net_revenue={float(silver_rev):,.2f}, rel_diff={rev_diff:.6%}"),
)

display(spark.createDataFrame(
    [("row_count", float(gold_rows), float(silver_rows), row_diff),
     ("net_revenue", float(gold_rev), float(silver_rev), rev_diff)],
    "metric string, gold double, silver double, rel_diff double",
))

In [0]:
# ─── DQ-06 (reported): rejected-record counts per table ───────────────────────────
# For each table: Bronze rows = Silver accepted + Silver quarantine rejects.
# Any gap would mean silent row loss, which is surfaced in the details.

reject_rows = []
for t in SOURCE_TABLES:
    bronze_n = spark.table(f"{BRONZE}.{t}").count()
    st = SILVER_TABLE_NAMES[t]
    silver_n = spark.table(f"{SILVER}.{st}").count()
    quar_n = spark.table(f"{SILVER}.{st}_quarantine").count()
    delta = bronze_n - silver_n
    reject_rows.append((t, bronze_n, silver_n, quar_n))
    record_check(
        "DQ-06", f"Rejected-record accounting — {t} (bronze = silver + quarantine)",
        "Silver", critical=False, passed=(delta == quar_n),
        details=(f"bronze={bronze_n:,}, silver={silver_n:,}, quarantine={quar_n:,}, "
                 f"bronze-silver={delta:,}"
                 + ("" if delta == quar_n else "  <-- MISMATCH: silent row loss!")),
    )

print("Rejected-record counts per table (Silver quarantine):")
display(spark.createDataFrame(
    reject_rows,
    "table string, bronze_rows long, silver_rows long, quarantine_rows long",
))

In [0]:
# ─── Final report & pipeline-stopping gate ─────────────────────────────────────
# Any CRITICAL check that failed raises here, stopping the pipeline.

total = len(check_results)
failed = [r for r in check_results if not r["passed"]]
critical_failures = [r for r in failed if r["critical"]]

summary_df = spark.createDataFrame(
    [(r["check_id"], r["layer"], "CRITICAL" if r["critical"] else "INFO",
      "PASS" if r["passed"] else "FAIL", r["description"], r["details"])
     for r in check_results],
    "check_id string, layer string, severity string, status string, description string, details string",
)

print(f"Data-quality checks executed: {total} | passed: {total - len(failed)} | "
      f"failed: {len(failed)} | critical failures: {len(critical_failures)}")
display(summary_df)

if critical_failures:
    lines = [f"  {r['check_id']} [{r['layer']}] {r['description']} — {r['details']}"
             for r in critical_failures]
    raise RuntimeError(
        "DATA QUALITY GATE FAILED — pipeline stopped. "
        f"{len(critical_failures)} critical check(s) violated:\n" + "\n".join(lines)
    )

print("ALL CRITICAL DATA QUALITY CHECKS PASSED — pipeline may proceed.")